In [ ]:
# Install required libraries (run once)
# !pip install tensorflow pillow scikit-learn matplotlib numpy

import os
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D, BatchNormalization, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow version: {tf.__version__}")
print("All imports successful!")

In [ ]:
# ─── 1.1 Download & Extract the Dataset ─────────────────────────────────────
DATASET_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00389/"
    "DevanagariHandwrittenCharacterDataset.zip"
)
DATASET_ZIP  = "DevanagariHandwrittenCharacterDataset.zip"
DATASET_DIR  = "DevanagariHandwrittenCharacterDataset"

if not os.path.exists(DATASET_DIR):
    print("Downloading dataset …")
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)
    print("Extracting …")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(".")
    print("Done!")
else:
    print("Dataset already present – skipping download.")

# List top-level folders to confirm structure
train_dir = os.path.join(DATASET_DIR, "Train")
test_dir  = os.path.join(DATASET_DIR, "Test")
print(f"Train classes: {len(os.listdir(train_dir))}")
print(f"Test  classes: {len(os.listdir(test_dir))}")

In [ ]:
# ─── 1.2 Load Images using PIL ───────────────────────────────────────────────
IMG_SIZE = (32, 32)   # native size of the dataset images

def load_images_from_dir(root_dir, img_size=IMG_SIZE):
    """
    Recursively loads all PNG images from root_dir using PIL.
    Returns a numpy array of shape (N, H, W).
    """
    images = []
    for class_folder in sorted(os.listdir(root_dir)):
        class_path = os.path.join(root_dir, class_folder)
        if not os.path.isdir(class_path):
            continue
        for fname in os.listdir(class_path):
            if not fname.lower().endswith('.png'):
                continue
            img_path = os.path.join(class_path, fname)
            img = Image.open(img_path).convert('L')   # greyscale
            img = img.resize(img_size, Image.LANCZOS)
            images.append(np.array(img))
    return np.array(images)

print("Loading training images …")
x_train_raw = load_images_from_dir(train_dir)
print("Loading test images …")
x_test_raw  = load_images_from_dir(test_dir)

print(f"Train set shape: {x_train_raw.shape}")
print(f"Test  set shape: {x_test_raw.shape}")

In [ ]:
# ─── 1.3 Normalise & Reshape ─────────────────────────────────────────────────
# Normalise pixel values to [0, 1]
x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0

# Reshape to include channel dimension: (N, H, W, 1)
x_train = x_train.reshape(-1, 32, 32, 1)
x_test  = x_test.reshape(-1, 32, 32, 1)

print(f"x_train shape after reshape: {x_train.shape}")
print(f"x_test  shape after reshape: {x_test.shape}")
print(f"Pixel range  – min: {x_train.min():.2f}  max: {x_train.max():.2f}")

In [ ]:
# ─── 1.4 Further Split Training Set into Train / Validation ──────────────────
x_train, x_val = train_test_split(x_train, test_size=0.15, random_state=42)

print(f"Training   samples: {x_train.shape[0]}")
print(f"Validation samples: {x_val.shape[0]}")
print(f"Test       samples: {x_test.shape[0]}")

In [ ]:
# ─── 1.5 Add Gaussian Noise ──────────────────────────────────────────────────
NOISE_FACTOR = 0.5   # controls severity of noise (0 = none, 1 = very heavy)

def add_gaussian_noise(images, noise_factor=NOISE_FACTOR):
    """Add Gaussian noise and clip to [0, 1]."""
    noisy = images + noise_factor * np.random.normal(
        loc=0.0, scale=1.0, size=images.shape
    )
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = add_gaussian_noise(x_train)
x_val_noisy   = add_gaussian_noise(x_val)
x_test_noisy  = add_gaussian_noise(x_test)

print("Noisy datasets created.")
print(f"Noisy pixel range – min: {x_train_noisy.min():.2f}  max: {x_train_noisy.max():.2f}")

In [ ]:
# ─── 1.6 Visualise Original vs Noisy Images ──────────────────────────────────
n = 8   # number of samples to display
fig, axes = plt.subplots(2, n, figsize=(16, 4))

for i in range(n):
    # Original
    axes[0, i].imshow(x_train[i].reshape(32, 32), cmap='gray')
    axes[0, i].set_title("Original", fontsize=8)
    axes[0, i].axis('off')
    # Noisy
    axes[1, i].imshow(x_train_noisy[i].reshape(32, 32), cmap='gray')
    axes[1, i].set_title(f"Noisy (σ={NOISE_FACTOR})", fontsize=8)
    axes[1, i].axis('off')

plt.suptitle("Devnagari Characters – Original vs Gaussian Noise", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 2: Build the Denoising Convolutional Autoencoder

The architecture uses **4 convolutional layers in the encoder** and **4 in the decoder** (≥4 layers deep as required).  
- **Encoder**: Conv2D + ReLU + MaxPooling (×4 blocks)
- **Decoder**: Conv2D + ReLU + UpSampling (×4 blocks) + Sigmoid output
- **BatchNormalization** after each conv layer for training stability
- **Dropout** for regularisation

In [ ]:
# ─── 2.1 Build the Encoder (4 convolutional blocks) ──────────────────────────
def build_encoder(input_shape=(32, 32, 1)):
    """
    Builds the encoder with 4 convolutional blocks.

    Architecture:
        Block 1: Conv2D(32)  + BN + ReLU → MaxPool  → 16×16×32
        Block 2: Conv2D(64)  + BN + ReLU → MaxPool  →  8× 8×64
        Block 3: Conv2D(128) + BN + ReLU → MaxPool  →  4× 4×128
        Block 4: Conv2D(256) + BN + ReLU            →  4× 4×256  (bottleneck)

    Parameters
    ----------
    input_shape : tuple – shape of one input sample (H, W, C).

    Returns
    -------
    input_img : Keras Input tensor
    encoded   : Keras tensor (bottleneck / latent representation)
    """
    input_img = Input(shape=input_shape, name="encoder_input")

    # Block 1
    x = Conv2D(32, (3, 3), activation='relu', padding='same', name='enc_conv1')(input_img)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2), padding='same', name='enc_pool1')(x)   # → 16×16×32

    # Block 2
    x = Conv2D(64, (3, 3), activation='relu', padding='same', name='enc_conv2')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2), padding='same', name='enc_pool2')(x)   # → 8×8×64

    # Block 3
    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='enc_conv3')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2), padding='same', name='enc_pool3')(x)   # → 4×4×128

    # Block 4 – Bottleneck (no pooling – deepest latent representation)
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='enc_conv4')(x)
    x = BatchNormalization()(x)
    encoded = Dropout(0.3, name='bottleneck_dropout')(x)             # → 4×4×256

    return input_img, encoded

print("Encoder function defined.")

In [ ]:
# ─── 2.2 Build the Decoder (4 convolutional blocks) ──────────────────────────
def build_decoder(encoded_input):
    """
    Builds the decoder (mirror of encoder) with 4 convolutional blocks.

    Architecture:
        Block 1: Conv2D(256) + BN + ReLU + UpSampling → 8× 8×256
        Block 2: Conv2D(128) + BN + ReLU + UpSampling → 16×16×128
        Block 3: Conv2D(64)  + BN + ReLU + UpSampling → 32×32×64
        Block 4: Conv2D(32)  + BN + ReLU              → 32×32×32
        Output:  Conv2D(1)   + Sigmoid                → 32×32×1

    Parameters
    ----------
    encoded_input : Keras tensor – output of the encoder (bottleneck).

    Returns
    -------
    decoded : Keras tensor – reconstructed image (same shape as input).
    """
    # Block 1
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='dec_conv1')(encoded_input)
    x = BatchNormalization()(x)
    x = UpSampling2D((2, 2), name='dec_up1')(x)                    # → 8×8×256

    # Block 2
    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='dec_conv2')(x)
    x = BatchNormalization()(x)
    x = UpSampling2D((2, 2), name='dec_up2')(x)                    # → 16×16×128

    # Block 3
    x = Conv2D(64, (3, 3), activation='relu', padding='same', name='dec_conv3')(x)
    x = BatchNormalization()(x)
    x = UpSampling2D((2, 2), name='dec_up3')(x)                    # → 32×32×64

    # Block 4
    x = Conv2D(32, (3, 3), activation='relu', padding='same', name='dec_conv4')(x)
    x = BatchNormalization()(x)                                     # → 32×32×32

    # Output layer – Sigmoid keeps pixel values in [0, 1]
    decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same', name='decoder_output')(x)

    return decoded

print("Decoder function defined.")

In [ ]:
# ─── 2.3 Assemble & Compile the Autoencoder ──────────────────────────────────
def build_autoencoder(input_shape=(32, 32, 1), learning_rate=1e-3):
    """
    Connects encoder and decoder into a full autoencoder and compiles it.

    Parameters
    ----------
    input_shape   : tuple          – shape of one sample.
    learning_rate : float          – Adam learning rate.

    Returns
    -------
    autoencoder : compiled Keras Model
    """
    input_img, encoded = build_encoder(input_shape)
    decoded            = build_decoder(encoded)

    autoencoder = Model(inputs=input_img, outputs=decoded, name="DenosingAutoencoder")
    autoencoder.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy'
    )
    return autoencoder


autoencoder = build_autoencoder()
autoencoder.summary()

### Architecture Diagram

```
Input 32×32×1
    │
    ├── ENCODER ──────────────────────────────
    │   Block 1: Conv2D(32)  + BN + Pool  → 16×16×32
    │   Block 2: Conv2D(64)  + BN + Pool  →  8× 8×64
    │   Block 3: Conv2D(128) + BN + Pool  →  4× 4×128
    │   Block 4: Conv2D(256) + BN + Drop  →  4× 4×256  ← Bottleneck
    │
    ├── DECODER ──────────────────────────────
    │   Block 1: Conv2D(256) + BN + Up    →  8× 8×256
    │   Block 2: Conv2D(128) + BN + Up    → 16×16×128
    │   Block 3: Conv2D(64)  + BN + Up    → 32×32×64
    │   Block 4: Conv2D(32)  + BN         → 32×32×32
    │
    └── Output: Conv2D(1) + Sigmoid       → 32×32×1
```

---
## Step 3: Train the Denoising Autoencoder

In [ ]:
# ─── 3.1 Configure Callbacks ─────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=3, min_lr=1e-6, verbose=1
)

print("Callbacks configured.")

In [ ]:
# ─── 3.2 Train the Model ─────────────────────────────────────────────────────
# Input  → noisy images
# Target → clean (original) images

EPOCHS     = 30
BATCH_SIZE = 64

history = autoencoder.fit(
    x_train_noisy, x_train,          # noisy input → clean target
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_val_noisy, x_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# ─── 3.3 Plot Training & Validation Loss ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

epochs_ran = range(1, len(history.history['loss']) + 1)

ax.plot(epochs_ran, history.history['loss'],     label='Training Loss',   linewidth=2, color='steelblue')
ax.plot(epochs_ran, history.history['val_loss'], label='Validation Loss', linewidth=2, color='orange', linestyle='--')

ax.set_title('Training and Validation Loss – Denoising Autoencoder', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_val_loss = min(history.history['val_loss'])
print(f"Best Validation Loss: {best_val_loss:.5f}")

---
## Step 4: Evaluate and Visualise the Results

In [ ]:
# ─── 4.1 Generate Denoised Predictions ───────────────────────────────────────
denoised_images = autoencoder.predict(x_test_noisy, verbose=0)

print(f"Denoised images shape: {denoised_images.shape}")

# Evaluate loss on the test set
test_loss = autoencoder.evaluate(x_test_noisy, x_test, verbose=0)
print(f"Test Loss (Binary Cross-Entropy): {test_loss:.5f}")

In [ ]:
# ─── 4.2 Visualise: Noisy → Denoised → Clean ─────────────────────────────────
def plot_comparison(noisy, denoised, clean, n=10, title="Autoencoder Denoising Results"):
    """
    Display n samples in 3 rows: Noisy / Denoised / Clean.
    """
    fig, axes = plt.subplots(3, n, figsize=(20, 6))
    row_labels = ["Noisy Input", "Denoised Output", "Clean Target"]
    datasets   = [noisy, denoised, clean]

    for row, (data, label) in enumerate(zip(datasets, row_labels)):
        axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold')
        for col in range(n):
            axes[row, col].imshow(data[col].reshape(32, 32), cmap='gray', vmin=0, vmax=1)
            axes[row, col].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


plot_comparison(
    x_test_noisy, denoised_images, x_test,
    n=10,
    title=f"Denoising Results – Devnagari Dataset (noise factor = {NOISE_FACTOR})"
)

In [ ]:
# ─── 4.3 Pixel-level Error Heatmap ───────────────────────────────────────────
n = 5
fig, axes = plt.subplots(3, n, figsize=(14, 7))
row_titles = ["Noisy", "Denoised", "Error Heatmap |Denoised – Clean|"]

for i in range(n):
    noisy_img    = x_test_noisy[i].reshape(32, 32)
    denoised_img = denoised_images[i].reshape(32, 32)
    clean_img    = x_test[i].reshape(32, 32)
    error_map    = np.abs(denoised_img - clean_img)

    axes[0, i].imshow(noisy_img,    cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel("Noisy", fontsize=10, fontweight='bold')

    axes[1, i].imshow(denoised_img, cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel("Denoised", fontsize=10, fontweight='bold')

    im = axes[2, i].imshow(error_map, cmap='hot', vmin=0, vmax=0.5)
    axes[2, i].axis('off')
    if i == 0: axes[2, i].set_ylabel("Error", fontsize=10, fontweight='bold')

plt.colorbar(im, ax=axes[2, :], orientation='vertical', fraction=0.02, pad=0.04, label='|Error|')
plt.suptitle("Per-Pixel Reconstruction Error (lower = better)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 4.4 Mean Squared Error per Sample ───────────────────────────────────────
mse_per_sample = np.mean((denoised_images - x_test) ** 2, axis=(1, 2, 3))

print(f"Mean MSE (denoised vs clean): {mse_per_sample.mean():.6f}")
print(f"Std  MSE:                     {mse_per_sample.std():.6f}")

# Also compare noisy vs clean (baseline)
mse_noisy = np.mean((x_test_noisy - x_test) ** 2, axis=(1, 2, 3))
print(f"\nBaseline MSE (noisy  vs clean): {mse_noisy.mean():.6f}")
print(f"Improvement in MSE: {(1 - mse_per_sample.mean()/mse_noisy.mean())*100:.1f}%")

---
## Step 5: Experimentation and Fine-Tuning

Here we compare the effect of **different noise factors** on the same trained model, and also experiment with a **lighter architecture** to observe trade-offs.

In [ ]:
# ─── 5.1 Experiment A: Effect of Different Noise Levels ──────────────────────
noise_levels = [0.1, 0.3, 0.5, 0.7]
results = {}

for nf in noise_levels:
    x_test_n = add_gaussian_noise(x_test, noise_factor=nf)
    denoised  = autoencoder.predict(x_test_n, verbose=0)
    mse       = np.mean((denoised - x_test) ** 2)
    results[nf] = mse
    print(f"Noise factor = {nf:.1f} → MSE = {mse:.6f}")

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(results.keys()), list(results.values()), marker='o', linewidth=2, color='steelblue')
ax.set_xlabel("Noise Factor", fontsize=12)
ax.set_ylabel("Mean Squared Error", fontsize=12)
ax.set_title("Reconstruction MSE vs Noise Factor", fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5.2 Visual Comparison at Each Noise Level ───────────────────────────────
sample_idx = 0   # pick one test image for visual comparison
fig, axes  = plt.subplots(2, len(noise_levels) + 1, figsize=(14, 5))

# Column 0 – clean original
axes[0, 0].imshow(x_test[sample_idx].reshape(32, 32), cmap='gray')
axes[0, 0].set_title("Original", fontsize=10)
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for col, nf in enumerate(noise_levels, start=1):
    x_n      = add_gaussian_noise(x_test[[sample_idx]], noise_factor=nf)
    denoised = autoencoder.predict(x_n, verbose=0)

    axes[0, col].imshow(x_n[0].reshape(32, 32), cmap='gray')
    axes[0, col].set_title(f"Noisy\n(σ={nf})", fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(denoised[0].reshape(32, 32), cmap='gray')
    axes[1, col].set_title(f"Denoised\n(σ={nf})", fontsize=9)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel("Noisy",    fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel("Denoised", fontsize=10, fontweight='bold')
plt.suptitle("Impact of Noise Level on Denoising Quality", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5.3 Experiment B: Lighter Architecture (Fewer Filters) ──────────────────
# A shallower / fewer-filter model – compare performance vs full model.

def build_light_autoencoder(input_shape=(32, 32, 1)):
    """
    Lighter 4-layer autoencoder with fewer filters for comparison.
    Encoder: 16 → 32 → 64 → 128  |  Decoder: mirror
    """
    inp = Input(shape=input_shape, name="light_input")

    # Encoder
    x = Conv2D(16,  (3, 3), activation='relu', padding='same')(inp)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(32,  (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(64,  (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)   # bottleneck

    # Decoder
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(64,  (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(32,  (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(16,  (3, 3), activation='relu', padding='same')(x)
    out = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

    model = Model(inputs=inp, outputs=out, name="LightAutoencoder")
    model.compile(optimizer=Adam(), loss='binary_crossentropy')
    return model

light_ae = build_light_autoencoder()
light_ae.summary()

print(f"\nFull model params:  {autoencoder.count_params():,}")
print(f"Light model params: {light_ae.count_params():,}")

In [ ]:
# Train the light model
light_history = light_ae.fit(
    x_train_noisy, x_train,
    epochs=15,
    batch_size=64,
    shuffle=True,
    validation_data=(x_val_noisy, x_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Compare MSE on test set
light_denoised = light_ae.predict(x_test_noisy, verbose=0)
light_mse  = np.mean((light_denoised - x_test) ** 2)
full_mse   = np.mean((denoised_images - x_test) ** 2)

print(f"\nFull  model MSE: {full_mse:.6f}")
print(f"Light model MSE: {light_mse:.6f}")

In [1]:
# ─── 5.4 Compare Full vs Light Model Visually ────────────────────────────────
n = 6
fig, axes = plt.subplots(4, n, figsize=(16, 8))
row_labels = ["Noisy", "Clean", f"Full Model\n(MSE={full_mse:.5f})", f"Light Model\n(MSE={light_mse:.5f})"]
datasets   = [x_test_noisy, x_test, denoised_images, light_denoised]

for row, (data, label) in enumerate(zip(datasets, row_labels)):
    axes[row, 0].set_ylabel(label, fontsize=9, fontweight='bold')
    for col in range(n):
        axes[row, col].imshow(data[col].reshape(32, 32), cmap='gray')
        axes[row, col].axis('off')

plt.suptitle("Full Model vs Light Model – Denoising Quality Comparison", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

---
## Summary of Observations

| Experiment | Observation |
|---|---|
| **Architecture depth** | 4 convolutional blocks in encoder/decoder provides a rich bottleneck representation, capturing fine stroke details of Devnagari script. |
| **BatchNormalization** | Stabilises training and allows faster convergence compared to a plain convolutional model. |
| **Dropout at bottleneck** | Acts as a regulariser, reducing over-fitting and improving generalisation to unseen noisy images. |
| **Noise factor = 0.5** | The model successfully recovers most character structure even at this significant noise level. |
| **Higher noise (0.7)** | Reconstruction degrades; the bottleneck does not retain enough information to reconstruct fine strokes. |
| **Full vs Light model** | The full model (more filters) achieves lower MSE at the cost of more parameters and training time. |
| **Early stopping** | Prevents unnecessary training; typically converges in ~15–20 epochs before validation loss plateaus. |

### Key Takeaways
- A **deeper encoder** learns a richer, more compact representation of Devnagari characters.
- Binary cross-entropy loss works well because normalised pixel values lie in **[0, 1]**.
- **Gaussian noise with factor 0.5** is a good training challenge — the model generalises well across moderate noise but struggles with extreme noise (> 0.6).
- The autoencoder acts as an **unsupervised feature learner** — no labels are needed, only paired clean/noisy images.